In [38]:
import pandas as pd
import numpy as np
import math
from datetime import timedelta

In [53]:
categories = pd.read_csv('end_to_end/datasets/categories_updated.csv')
cities = pd.read_csv('end_to_end/datasets/cities_realistic.csv')
customers = pd.read_csv('end_to_end/datasets/customers_updated.csv')
employees = pd.read_csv('end_to_end/datasets/employees_expanded.csv')
products = pd.read_csv('end_to_end/datasets/products_updated.csv')
sales = pd.read_csv('end_to_end/datasets/sales_6m.csv')

In [55]:
products.head()

,product_id,product_name,resistant,vitality_days,category_id,reorder_levels,stock_quantity,cost_price,selling_price,seasonal_flag,demand_score,category_name,margin_percentage,lead_time_days
0,1,Flour - Whole Wheat,Durable,321,7,400.0,644,8.9,10.6,False,92,Grains & Cereals,9.57,13
1,2,Cookie Chocolate Chip With,Weak,68,2,300.0,2688,22.4,30.6,False,84,Bakery & Pastries,33.67,2
2,3,Onions - Cippolini,Weak,302,16,200.0,4368,17.3,18.8,False,91,Vegetables,12.95,1
3,4,"Sauce - Gravy, Au Jus, Mix",Durable,245,4,250.0,684,14.1,17.3,False,36,Condiments & Sauces,24.16,9
4,5,Artichokes - Jerusalem,Weak,305,16,150.0,2520,3.8,4.0,False,56,Vegetables,12.95,3


In [54]:
sales.head()

,sales_id,employee_id,customer_id,product_id,quantity,sales_date,transaction_number
0,10527,140,91792,2,1,2024-01-01 00:00:02,3c2a68b5-8e4c-40e1-a4b5-425ed2926966
1,3520,84,45889,214,5,2024-01-01 00:00:07,af728480-d8ee-4ada-b91b-e861dd1a777b
2,13385,80,20981,257,2,2024-01-01 00:00:10,c05adb1c-cdf4-445f-a94f-ad53d5b1a6b2
3,2562,145,20317,405,18,2024-01-01 00:00:13,8f0800f1-eb48-4f6f-88a7-51d4cf894d43
4,4070,229,28242,246,7,2024-01-01 00:00:22,70213f79-4f20-4c3e-955e-48f405ba8f72


In [40]:
sales['sales_date'] = pd.to_datetime(sales['sales_date'])
# Removing transaction ID coz its not important for now.
transaction_number = sales['transaction_number']
sales.drop(columns='transaction_number', inplace=True)
# Reseting the sales_id to an increasing integer list.
sales['sales_id'] = range(1, len(sales)+1)

In [41]:
# Function to set a week start tag for every week in the sales data
def week_start_date(dt_series):
    weekday = dt_series.dt.weekday
    week_start = (dt_series - pd.to_timedelta(weekday, unit='d')).dt.normalize().dt.date
    return week_start

In [42]:
def divide_into_months(dt_series):
    month_number = dt_series.dt.month
    return month_number

In [43]:
sales['week_start_date'] = week_start_date(sales['sales_date'])
sales['month_number'] = divide_into_months(sales['sales_date'])

In [44]:
sales.head()

,sales_id,employee_id,customer_id,product_id,quantity,sales_date,week_start_date,month_number
0,1,140,91792,2,1,2024-01-01 00:00:02,2024-01-01,1
1,2,84,45889,214,5,2024-01-01 00:00:07,2024-01-01,1
2,3,80,20981,257,2,2024-01-01 00:00:10,2024-01-01,1
3,4,145,20317,405,18,2024-01-01 00:00:13,2024-01-01,1
4,5,229,28242,246,7,2024-01-01 00:00:22,2024-01-01,1


In [45]:
sales.groupby(['week_start_date', 'product_id'])['quantity'].sum().reset_index(name='total_weekly_sales')

,week_start_date,product_id,total_weekly_sales
0,2024-01-01,1,1433
1,2024-01-01,2,496
2,2024-01-01,3,2759
3,2024-01-01,4,1403
4,2024-01-01,5,1220
...,...,...,...
23951,2024-12-30,448,126
23952,2024-12-30,449,413
23953,2024-12-30,450,902
23954,2024-12-30,451,153


In [47]:
# Processing sales data to extract different statistical analytics from the data (mostly sales averages over different timelines)
def process_sales_df(sales_df, lookback_weeks=4):

    for c in ('product_id', 'quantity', 'sales_date'):
        if c not in sales_df.columns:
            raise ValueError(f"Sales DataFrame must contain '{c}' column.")
    
    sales = sales_df.copy()
    sales['product_id'] = sales['product_id'].astype(str)
    sales['quantity'] = pd.to_numeric(sales['quantity'], errors='coerce').fillna(0)
    sales['sales_date'] = pd.to_datetime(sales['sales_date'], errors='coerce')

    sales = sales[sales['sales_date'].notna() & (sales['sales_date'].notna())]

    overall_latest = sales['sales_date'].max()
    if pd.isna(overall_latest):
        raise ValueError('No valid sales_date found in sales data.')
    
    sales['week_start'] = week_start_date(sales['sales_date'])

    agg = sales.groupby('product_id').agg(
        total_qty=('quantity', 'sum'),
        first_date=('sales_date', 'min'),
        last_date=('sales_date', 'max')
    ).reset_index()

    # Only use when working on real data
    # agg['first_date'] = pd.to_datetime(agg['first_date'], errors='coerce')
    # agg['last_date']  = pd.to_datetime(agg['last_date'], errors='coerce')
    # agg['days_active'] = (agg['last_date'].dt.date - agg['first_date'].dt.date)
    # agg['days_active'] = agg['days_active'].days
    # agg['days_active'] = agg['days_active'].clip(lower=1)
    agg['weekly_avg'] = agg['total_qty'] / int(365 / 7.0)

    weekly = sales.groupby(['product_id', 'week_start'], observed=True)['quantity'].sum().reset_index()

    overall_latest_date = overall_latest.date()
    lookback_threshold = overall_latest_date - timedelta(days=lookback_weeks * 7 - 1)
    recent_weekly = (
        weekly[weekly['week_start'] >=lookback_threshold]
        .groupby('product_id')['quantity'].sum().rename('recent_total_qty').reset_index()
    )
    recent_weekly['recent_weekly_avg'] = recent_weekly['recent_total_qty'] / lookback_weeks
    metrics_df = agg.merge(recent_weekly[['product_id', 'recent_weekly_avg']], on='product_id', how='left')
    metrics_df['recent_weekly_avg'] = metrics_df['recent_weekly_avg'].fillna(0.0)

    return metrics_df, weekly

In [48]:
# Computing the demand score for every product based on trends in different timelines
def compute_demand_score(metrics_df, w_weekly=0.7, w_trend=0.3):
    df = metrics_df.copy()
    df['trend'] = df.apply(
        lambda r: (r['recent_weekly_avg'] / r['weekly_avg']) if r['weekly_avg'] > 0 else (r['recent_weekly_avg'] if r['recent_weekly_avg'] > 0 else 0.0),
        axis=1
    )
    def minmax(s):
        minv, maxv = s.min(), s.max()
        if pd.isna(minv) or pd.isna(maxv) or minv == maxv:
            return pd.Series(0.0, index=s.index)
        return (s - minv) / (maxv - minv)
    
    df['weekly_norm'] = minmax(df['weekly_avg'])
    df['trend_norm'] = minmax(df['trend'])
    df['demand_score'] = (w_weekly *df['weekly_norm'] + w_trend * df['trend_norm']) * 100.0
    df['demand_score'] = df['demand_score'].clip(0.0, 100.0)
    return df

In [49]:

def compute_reorder_levels_and_quantities(demand_df, products_df, default_vitality_days=30):
    prod = products_df.copy()
    prod["product_id"] = prod["product_id"].astype(str)

    if "vitality_days" not in prod.columns:
        prod["vitality_days"] = default_vitality_days

    prod["vitality_days"] = pd.to_numeric(prod["vitality_days"], errors="coerce").fillna(default_vitality_days).astype(int)

    has_stock = "stock_quantity" in prod.columns
    
    if has_stock:
        prod["stock_quantity"] = pd.to_numeric(prod["stock_quantity"], errors="coerce").fillna(0).astype(int)

    merged = demand_df.merge(prod[["product_id", "vitality_days"] + (["stock_quantity"] if has_stock else [])],
                             on="product_id", how="left")

    merged["avg_daily_demand"] = merged["recent_weekly_avg"] / 7.0

    merged["reorder_level"] = (np.ceil(merged["avg_daily_demand"] * merged["vitality_days"])).astype(int)

    if has_stock:
        merged["reorder_quantity"] = (merged["reorder_level"] - merged["stock_quantity"]).clip(lower=0).astype(int)
    else:
        merged["reorder_quantity"] = merged["reorder_level"]

    return merged

In [50]:
restock_avg_per_product , weekly_data = process_sales_df(sales)
restock_avg_per_product

,product_id,total_qty,first_date,last_date,weekly_avg,recent_weekly_avg
0,1,72646,2024-01-01 00:17:05,2024-12-31 22:17:12,1397.038462,1179.75
1,10,26585,2024-01-01 01:19:13,2024-12-31 23:53:28,511.250000,433.25
2,100,73631,2024-01-01 00:02:05,2024-12-31 23:50:15,1415.980769,1153.00
3,101,26103,2024-01-01 00:39:43,2024-12-31 22:55:27,501.980769,396.25
4,102,73481,2024-01-01 00:49:12,2024-12-31 22:55:46,1413.096154,1232.50
...,...,...,...,...,...,...
447,95,72094,2024-01-01 00:13:18,2024-12-31 23:33:13,1386.423077,1158.75
448,96,73205,2024-01-01 00:29:17,2024-12-31 23:39:40,1407.788462,1110.50
449,97,141023,2024-01-01 02:14:35,2024-12-31 23:54:37,2711.980769,2012.00
450,98,26787,2024-01-01 00:04:47,2024-12-31 23:18:49,515.134615,424.25


In [51]:
demand_score = compute_demand_score(metrics_df=restock_avg_per_product)

In [52]:
compute_reorder_levels_and_quantities(demand_score, products).drop(columns=['reorder_level', 'reorder_quantity', 'first_date', 'last_date']).head()

,product_id,total_qty,weekly_avg,recent_weekly_avg,trend,weekly_norm,trend_norm,demand_score,vitality_days,stock_quantity,avg_daily_demand
0,1,72646,1397.038462,1179.75,0.844465,0.400784,0.676770,48.357967,321,644,168.535714
1,10,26585,511.250000,433.25,0.847433,0.007497,0.692640,21.303965,99,3640,61.892857
2,100,73631,1415.980769,1153.00,0.814277,0.409194,0.515344,44.103899,323,663,164.714286
3,101,26103,501.980769,396.25,0.789373,0.003381,0.382176,11.701957,216,1900,56.607143
4,102,73481,1413.096154,1232.50,0.872198,0.407913,0.825069,53.305993,64,975,176.071429
